In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from darts import TimeSeries
from darts.models import BlockRNNModel
from darts.dataprocessing.transformers import Scaler
import torch.nn as nn
from narwhals import DataFrame

In [ ]:
# ==========================================
# 1. Data Loading
# ==========================================
FILE_PATH = '../data_for_prediction/ID_10.csv'

df = pd.read_csv(
    FILE_PATH,
    sep=',',
)

df['Date'] = pd.to_datetime(df[['YYYY', 'MM', 'DD']].rename(columns={
    'YYYY': 'year',
    'MM': 'month',
    'DD': 'day'
}))

# Set the index to the new Date column
df = df.set_index('Date')

# drop data columns
df.drop(columns=['YYYY', 'MM', 'DD', 'DOY'], inplace=True)

print(df.head())

In [ ]:
# ==========================================
# 2. Feature Separation (Target vs. Covariates)
# ==========================================
TARGET_VAR = 'prec'

# 1. Create the Target Series (What we want to predict)
target_series = TimeSeries.from_series(df[TARGET_VAR], freq='D')

# 2. Create the Covariates Series (All OTHER features)
# We drop the target column to create a dataframe of just features
covariates_df = df.drop(columns=[TARGET_VAR])

# Convert features to Darts TimeSeries
covariates_series = TimeSeries.from_dataframe(covariates_df, freq='D')

print(f"Target variable: {TARGET_VAR}")
print(f"Covariates (Features) being used: {covariates_series.components.tolist()}")

In [ ]:
# ==========================================
# 3. Preprocessing (Scaling)
# ==========================================
# Neural networks require scaling. We must scale Target and Covariates separately.
target_scaler = Scaler()
cov_scaler = Scaler()

# Fit and transform
target_scaled = target_scaler.fit_transform(target_series)
cov_scaled = cov_scaler.fit_transform(covariates_series)

# Split into train/validation (80/20 split)
train_target, val_target = target_scaled.split_before(0.8)
train_cov, val_cov = cov_scaled.split_before(0.8)

# ==========================================
# 4. Model Definition (Multivariate)
# ==========================================
model = BlockRNNModel(
    input_chunk_length=30,  # Look back 30 days
    output_chunk_length=7,  # Predict 7 days
    model='LSTM',
    hidden_dim=20,
    n_rnn_layers=1,
    n_epochs=15,            # Increased epochs slightly for more complex data
    dropout=0.1,
    batch_size=16,
    random_state=42,
    loss_fn=nn.L1Loss() # better for precipitation because it is less sensitive to extreme outliers and discourages the model from "blurring" predictions (predicting constant drizzle to minimize squared error)
)

# ==========================================
# 5. Fitting with Covariates
# ==========================================
# We pass 'past_covariates' here. The model learns:
# [Past Rain + Past Temp + Past Flow] -> [Future Rain]
print("Fitting multivariate model...")
model.fit(
    series=train_target,
    past_covariates=train_cov,
    val_series=val_target,
    val_past_covariates=val_cov,
    verbose=True
)

In [ ]:
# ==========================================
# 6. Prediction
# ==========================================
# We predict 7 days into the future from the end of the training set.
# We must provide the covariates so the model can see the "recent past" features.
pred_scaled = model.predict(
    n=7,
    series=train_target,
    past_covariates=cov_scaled # Pass FULL covariates (train+val) for context
)

# Inverse transform to get real units (
# mm)
pred = target_scaler.inverse_transform(pred_scaled)

# ==========================================
# 7. Extract Results
# ==========================================
day_1 = pred[0].values()[0][0]
day_3 = pred[2].values()[0][0]
day_7 = pred[6].values()[0][0]

print("\n--- Forecast Results (Precipitation) using All Features ---")
print(f"1-Day Forecast: {day_1:.4f} mm")
print(f"3-Day Forecast: {day_3:.4f} mm")
print(f"7-Day Forecast: {day_7:.4f} mm")

In [ ]:
# Plotting
plt.figure(figsize=(10, 6))
# Plot actual validation data (unscaled) for comparison
val_target_unscaled = target_scaler.inverse_transform(val_target)
val_target_unscaled[:30].plot(label='Actual Validation Data') # Plot first 30 days of val
pred.plot(label='7-Day Forecast', color='red')
plt.title(f'Multivariate Prediction (Using {len(covariates_series.components)} features)')
plt.legend()
plt.show()